Candidate inspection: Moon and Earth

Diagnose why Wadi Rum's steep patches supply few lunar counterparts, and what
Arizona contributes under diagnostic land-cover policies. Follow the
declared inspection described below. Existing masks and physical
bands remain unchanged. No model, reserve-region inspection or Mars data.

In [ ]:
import hashlib
import json
import sys
from datetime import UTC, datetime
from importlib.metadata import version
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import rasterio
from IPython.display import Image, Markdown, display
from matplotlib.figure import Figure
from rasterio.warp import transform as project_points
from rasterio.warp import transform_bounds
from rasterio.windows import Window, from_bounds

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'outline.md').exists())
sys.path.insert(0, str(ROOT / 'notebooks'))
from candidate_analysis import terrain_band_components
from inspection import check_input, file_hash, verify_run
from retrieval_analysis import terrain_discrepancies

NOTEBOOK = ROOT / 'notebooks/exploration/23_candidate_inspection_eda.ipynb'
PREVIOUS = ROOT / 'notebooks/diagnostics/screening-eda-20261029T091841917085Z'
SCALE = ROOT / 'notebooks/diagnostics/scale-eda-20261029T012946475712Z'
MOON = ROOT / 'data/processed/lunar_terrain_views/20261028T084443441751Z_1a794f'
OUT = ROOT / 'notebooks/diagnostics' / ('candidate-inspection-' + datetime.now(UTC).strftime('%Y%m%dT%H%M%S%fZ'))
OUT.mkdir(parents=True, exist_ok=False)
FEATURES = ['slope_median_deg', 'slope_p90_deg', 'central_relief_m', 'plane_residual_fraction', 'local_sd_median_m']
COMPONENTS = ['median', 'p90', 'relief', 'planarity']
POLICIES = ['baseline', 'patch_mean_bare80', 'no_bare_requirement']
SITES = ['atacama_san_pedro', 'northern_arizona', 'wadi_rum']
SCENARIOS = ['baseline', 'with_planarity']
expected: dict[str, str] = {}
evidence: dict[str, str] = {}
for previous in [PREVIOUS, SCALE]:
    manifest = json.loads((previous / 'provenance.json').read_text())
    assert manifest['status'] == 'complete'
    expected.update({str((previous / name).resolve()): digest for name, digest in manifest['outputs'].items()})
    evidence[str(previous / 'provenance.json')] = file_hash(previous / 'provenance.json')
screening_manifest = json.loads((PREVIOUS / 'provenance.json').read_text())
for path in [MOON, *[ROOT / p for p in screening_manifest['runs'].values()]]:
    expected.update(verify_run(path))
    evidence[str(path / 'provenance.json')] = file_hash(path / 'provenance.json')
# Previous consumed native classes are also pinned by their raw/processed hashes.
expected.update(screening_manifest['inputs'])


def checked(path: Path) -> Path:
    """Hash each immutable input once per execution before its first read."""
    if str(path.resolve()) not in evidence:
        check_input(path, expected, evidence)
    return path


earth = pd.read_csv(checked(PREVIOUS / 'center_screening_metrics.csv.gz'))
earth['unit_id'] = earth['center_id'] + ':' + earth['size_m'].astype(str)
earth = earth.sort_values(['region', 'size_m', 'center_row', 'center_col']).reset_index(drop=True)
earth_lookup = earth.set_index('unit_id', verify_integrity=True)
all_metrics = pd.read_csv(checked(SCALE / 'center_metrics.csv'))
queries = all_metrics.loc[(all_metrics['body'] == 'moon') & all_metrics['common']].copy()
queries['query_id'] = queries['center_id'] + ':' + queries['size_m'].astype(str)
query_lookup = queries.set_index('query_id', verify_integrity=True)
prior = pd.read_csv(checked(PREVIOUS / 'query_policy_results.csv.gz'))
prior = prior.loc[prior['cohort'] == 'eligible_at_size'].set_index(['query_id', 'policy', 'scenario'], verify_integrity=True)
grids = json.loads(checked(PREVIOUS / 'site_grids.json').read_text())
source = '\n'.join(c.source for c in nbformat.read(NOTEBOOK, as_version=4).cells)
provenance: dict[str, Any] = {
    'status': 'running', 'notebook': str(NOTEBOOK.relative_to(ROOT)),
    'notebook_source_sha256': hashlib.sha256(source.encode()).hexdigest(),
    'previous_eda': str(PREVIOUS.relative_to(ROOT)), 'scale_eda': str(SCALE.relative_to(ROOT)),
    'moon_run': str(MOON.relative_to(ROOT)), 'role': 'development only; Moon and Earth',
    'earth_cohort': 'eligible_at_size', 'lunar_cohort': 'fixed notebook16 common960',
    'sizes_m': [300, 600, 960], 'features': FEATURES, 'components': COMPONENTS,
    'pools': 'Arizona/Wadi all3policies; Atacama baseline control',
    'helpers': {name: file_hash(ROOT / 'notebooks' / name) for name in
                ['candidate_analysis.py', 'retrieval_analysis.py', 'inspection.py']},
    'software': {p: version(p) for p in ['numpy', 'pandas', 'rasterio', 'matplotlib', 'nbformat']},
}
(OUT / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')


def save_figure(fig: Figure, name: str) -> None:
    fig.savefig(OUT / f'{name}.png', bbox_inches='tight')
    fig.savefig(OUT / f'{name}.pdf', bbox_inches='tight')
    display(Image(filename=str(OUT / f'{name}.png')))
    plt.close(fig)


print('Output:', OUT)

# Marginal versus joint support and deterministic nearest candidates

A pool can contain a candidate for each individual band while containing none
that meet them together. Inspect both cases. Scores and counts must reproduce
notebook22 exactly for the same cohort, policy, site and query.

In [ ]:
pair_records: list[dict[str, Any]] = []
for size in [300, 600, 960]:
    groups = {site: earth.loc[(earth['region'] == site) & (earth['size_m'] == size)].reset_index(drop=True) for site in SITES}
    for query in queries.loc[queries['size_m'] == size].to_dict('records'):
        q = np.array([query[f] for f in FEATURES], dtype=float)
        for site, group in groups.items():
            values = group[FEATURES].to_numpy(dtype=float)
            components = terrain_band_components(q, values)
            canonical = terrain_discrepancies(q, values)
            np.testing.assert_allclose(components[:, :3].max(axis=1), canonical['baseline'], rtol=1e-12)
            np.testing.assert_allclose(components.max(axis=1), canonical['with_planarity'], rtol=1e-12)
            for policy in (['baseline'] if site == 'atacama_san_pedro' else POLICIES):
                selected = np.flatnonzero(group[policy].to_numpy())
                assert selected.size > 0
                part = components[selected]
                passes = part <= 1
                record = {'query_id': query['query_id'], 'region': query['region'], 'population': query['population'],
                          'size_m': size, 'earth_site': site, 'policy': policy, 'candidate_count': len(selected)}
                for j, field in enumerate(COMPONENTS):
                    record[f'{field}_count'] = int(np.count_nonzero(passes[:, j]))
                for j, field in enumerate(COMPONENTS[:3]):
                    record[f'without_{field}_count'] = int(np.count_nonzero(np.delete(passes[:, :3], j, axis=1).all(axis=1)))
                for scenario, prefix, included in [('baseline', 'b', 3), ('with_planarity', 'p', 4)]:
                    scores = part[:, :included].max(axis=1)
                    winner_position = int(np.argmin(scores))
                    winner = group.iloc[int(selected[winner_position])]
                    count, best = int(np.count_nonzero(scores <= 1)), float(scores[winner_position])
                    old = prior.loc[(query['query_id'], policy, scenario)]
                    assert old[f'{site}_count'] == count
                    np.testing.assert_allclose(old[f'{site}_best'], best, rtol=1e-12, atol=1e-12)
                    record[f'{scenario}_count'] = count
                    record[f'{prefix}_score'] = best
                    record[f'{prefix}_candidate_id'] = winner['unit_id']
                    for feature in FEATURES:
                        record[f'{prefix}_{feature}'] = float(winner[feature])
                    for j, component in enumerate(COMPONENTS):
                        record[f'{prefix}_d_{component}'] = float(part[winner_position, j])
                pair_records.append(record)
    print('Diagnosed', size, 'm')
pairs = pd.DataFrame(pair_records)
assert not pairs.duplicated(['query_id', 'earth_site', 'policy']).any()
pairs.to_csv(OUT / 'candidate_diagnostics.csv.gz', index=False, compression='gzip')
pair_lookup = pairs.set_index(['query_id', 'earth_site', 'policy'], verify_integrity=True)
COUNTS = [f'{x}_count' for x in COMPONENTS] + [f'without_{x}_count' for x in COMPONENTS[:3]] + ['baseline_count', 'with_planarity_count']
summary_rows = []
KEYS = ['size_m', 'population', 'earth_site', 'policy']
for key, group in pairs.groupby(KEYS):
    assert isinstance(key, tuple)
    for region, part in [('ALL_QUERY_WEIGHTED', group), *list(group.groupby('region'))]:
        row = {**dict(zip(KEYS, key, strict=True)), 'region': region, 'queries': len(part)}
        row.update({field.removesuffix('_count') + '_pct': float(100 * (part[field] > 0).mean()) for field in COUNTS})
        row['baseline_best_median'] = float(part['b_score'].median())
        row['planarity_best_median'] = float(part['p_score'].median())
        row['winner_changed_pct'] = float(100 * (part['b_candidate_id'] != part['p_candidate_id']).mean())
        summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT / 'constraint_coverage.csv', index=False)
pct_fields = [f for f in summary if f.endswith('_pct')]
macro = summary.loc[summary['region'] != 'ALL_QUERY_WEIGHTED'].groupby(KEYS)[pct_fields].mean().reset_index()
macro.to_csv(OUT / 'constraint_coverage_region_macro.csv', index=False)

# Distribution and constraint checks

Source support and spatially varying errors can still affect these estimated
surfaces. These plots diagnose descriptor support, not independent analog truth.

In [ ]:
plot_fields = ['median_pct', 'p90_pct', 'relief_pct', 'without_relief_pct', 'without_p90_pct',
               'without_median_pct', 'baseline_pct', 'with_planarity_pct']
plot_labels = ['Median slope', 'P90 slope', 'Relief', 'Median + P90', 'Median + relief', 'P90 + relief', 'All three', '+ Planarity']
fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharey=True)
for i, (site, policy) in enumerate([('wadi_rum', 'baseline'), ('wadi_rum', 'no_bare_requirement'), ('northern_arizona', 'no_bare_requirement')]):
    for j, population in enumerate(['full_context', 'image_supported']):
        row = summary.loc[(summary['size_m'] == 600) & (summary['region'] == 'ALL_QUERY_WEIGHTED')
                          & (summary['earth_site'] == site) & (summary['policy'] == policy) & (summary['population'] == population)].iloc[0]
        axes[i, j].bar(np.arange(len(plot_fields)), row[plot_fields].to_numpy(dtype=float), color='#24758a')
        axes[i, j].set(title=f'{site} / {policy}\n{population}', ylabel='Queries with any candidate (%)', ylim=(0, 105),
                       xticks=np.arange(len(plot_fields)), xticklabels=plot_labels)
        axes[i, j].tick_params(axis='x', labelrotation=35, labelsize=8)
fig.suptitle('600 m: marginal support does not establish joint support')
fig.tight_layout(h_pad=2)
save_figure(fig, 'constraint_support')

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
lunar = queries.loc[(queries['size_m'] == 600) & (queries['population'] == 'image_supported')]
for i, (site, policy) in enumerate([('wadi_rum', 'baseline'), ('northern_arizona', 'no_bare_requirement')]):
    subset = earth.loc[(earth['region'] == site) & (earth['size_m'] == 600) & earth[policy]]
    for j, (x, y) in enumerate([('slope_median_deg', 'slope_p90_deg'), ('central_relief_m', 'slope_p90_deg'),
                                ('plane_residual_fraction', 'slope_p90_deg')]):
        axes[i, j].scatter(subset[x], subset[y], s=4, alpha=.2, label=site, rasterized=True)
        axes[i, j].scatter(lunar[x], lunar[y], s=5, alpha=.35, label='Moon strict', rasterized=True)
        axes[i, j].set(xlabel=x, ylabel=y, title=f'{site}: {policy}')
    axes[i, 0].legend(fontsize=8)
fig.suptitle('600 m descriptor arrangements; points are spatially dependent')
fig.tight_layout()
save_figure(fig, 'descriptor_arrangements')

# Declared example selection

One query per lunar region, then at most two supplementary controls. Their
outcome-based selection is explicit; they do not estimate retrieval accuracy.

In [ ]:
strict = queries.loc[(queries['size_m'] == 600) & (queries['population'] == 'image_supported')].copy()
focus = pairs.loc[(pairs['size_m'] == 600) & (pairs['population'] == 'image_supported')]
az_base = focus.loc[(focus['earth_site'] == 'northern_arizona') & (focus['policy'] == 'baseline')].set_index('query_id')
az_free = focus.loc[(focus['earth_site'] == 'northern_arizona') & (focus['policy'] == 'no_bare_requirement')].set_index('query_id')
wadi = focus.loc[(focus['earth_site'] == 'wadi_rum') & (focus['policy'] == 'baseline')].set_index('query_id')
selected_queries: list[dict[str, Any]] = []


def choose_example(ids: list[str], scores: pd.Series, reason: str) -> None:
    """Choose the center closest to the group's median, with a stable query-ID tie break."""
    if not ids:
        return
    frame = scores.loc[ids].rename('score').reset_index()
    frame['distance'] = abs(frame['score'] - frame['score'].median())
    winner = frame.sort_values(['distance', 'query_id']).iloc[0]
    if winner['query_id'] not in {r['query_id'] for r in selected_queries}:
        selected_queries.append({'query_id': winner['query_id'], 'selection_reason': reason,
                                 'group_queries': len(ids), 'selection_score': winner['score']})


for region, group in strict.groupby('region'):
    ids = group['query_id'].tolist()
    groups_for_selection = [
        ('new_arizona_with_planarity', (az_base['baseline_count'] == 0) & (az_free['with_planarity_count'] > 0)),
        ('new_arizona_baseline_only', (az_base['baseline_count'] == 0) & (az_free['baseline_count'] > 0)),
        ('wadi_baseline_failure', wadi['baseline_count'] == 0),
        ('region_fallback', pd.Series(True, index=az_free.index)),
    ]
    for reason, flag in groups_for_selection:
        candidates = [q for q in ids if bool(flag.loc[q])]
        if candidates:
            choose_example(candidates, az_free['b_score'], f'{region}:{reason}')
            break
choose_example(wadi.index[wadi['baseline_count'] > 0].tolist(), wadi['b_score'], 'supplement:Wadi baseline match')
choose_example(az_free.index[az_free['baseline_count'] == 0].tolist(), az_free['b_score'], 'supplement:Arizona no-bare failure')
selections = pd.DataFrame(selected_queries)
assert 7 <= len(selections) <= 9
selections.to_csv(OUT / 'gallery_queries.csv', index=False)


def read_patch(unit: dict[str, Any], channel: str, body: str) -> tuple[np.ndarray, Any, Any]:
    if body == 'earth':
        folder = ROOT / grids[unit['region']]['folder']
    else:
        folder = MOON / 'moon' / unit['region'] / unit['population']
    path = checked(folder / f'{channel}.tif')
    side = int(unit['size_m']) // 30
    r, c = int(unit['row']), int(unit['col'])
    with rasterio.open(path) as src:
        values = src.read(1, window=((r, r + side), (c, c + side)), masked=True).astype('float64').filled(np.nan)
        assert values.shape == (side, side) and np.isfinite(values).all()
        return values, src.transform, src.crs


def plane_residual(values: np.ndarray) -> np.ndarray:
    """Least-squares residual for display, retaining meters and original orientation."""
    yy, xx = np.indices(values.shape)
    design = np.column_stack([np.ones(values.size), xx.ravel(), yy.ravel()])
    residual = values.ravel() - design @ np.linalg.lstsq(design, values.ravel(), rcond=None)[0]
    return residual.reshape(values.shape)


gallery_records = []
gallery_candidate_ids: set[str] = set()
for number, selected in enumerate(selected_queries, 1):
    query_id = selected['query_id']
    query = query_lookup.loc[query_id].to_dict()
    panels = [('Moon query', query, 0., 'moon', query_id)]
    for title, site, policy, prefix in [('Atacama baseline', 'atacama_san_pedro', 'baseline', 'b'),
                                       ('Wadi baseline', 'wadi_rum', 'baseline', 'b'),
                                       ('Arizona no-bare', 'northern_arizona', 'no_bare_requirement', 'b'),
                                       ('Arizona + planarity', 'northern_arizona', 'no_bare_requirement', 'p')]:
        match = pair_lookup.loc[(query_id, site, policy)]
        candidate_id = str(match[f'{prefix}_candidate_id'])
        panels.append((title, earth_lookup.loc[candidate_id].to_dict(), float(match[f'{prefix}_score']), 'earth', candidate_id))
        gallery_candidate_ids.add(candidate_id)
    arrays = []
    for title, unit, score, body, unit_id in panels:
        z, transform, crs = read_patch(unit, 'elevation_m', body)
        slope, other_transform, other_crs = read_patch(unit, 'slope_deg', body)
        assert transform == other_transform and crs == other_crs
        residual = plane_residual(z)
        rms = float(np.sqrt(np.mean(residual ** 2)))
        np.testing.assert_allclose(rms, unit['plane_residual_rms_m'], atol=1e-8, rtol=1e-8)
        arrays.append((z - z.mean(), slope, residual))
        gallery_records.append({'gallery': number, 'query_id': query_id, 'panel': title, 'body': body,
                                'unit_id': unit_id, 'region': unit['region'], 'score': score,
                                'center_x_m': unit['center_x_m'], 'center_y_m': unit['center_y_m'],
                                'crs_wkt': crs.to_wkt(), 'row': unit['row'], 'col': unit['col'],
                                **{f: unit[f] for f in FEATURES}, 'plane_residual_rms_m': rms})
    height_limit = max(float(np.max(np.abs(a[0]))) for a in arrays)
    slope_limit = max(float(np.max(a[1])) for a in arrays)
    residual_limit = max(.01, max(float(np.max(np.abs(a[2]))) for a in arrays))
    limits = [(-height_limit, height_limit), (0., slope_limit), (-residual_limit, residual_limit)]
    fig, axes = plt.subplots(3, 5, figsize=(18, 10), layout='constrained')
    for col, ((title, unit, score, _, _), layers) in enumerate(zip(panels, arrays, strict=True)):
        for row, (values, (low, high), label, cmap) in enumerate(zip(layers, limits,
                    ['Mean-centered elevation (m)', 'Slope (degrees)', 'Plane residual (m)'], ['RdBu_r', 'viridis', 'RdBu_r'], strict=True)):
            im = axes[row, col].imshow(values, extent=(0, 600, 0, 600), vmin=low, vmax=high, cmap=cmap, interpolation='nearest')
            axes[row, col].set(xticks=[0, 300, 600], yticks=[0, 300, 600], xlabel='East (m)')
            if col == 0:
                axes[row, col].set_ylabel(label + '\nNorth (m)')
            if row == 0:
                axes[row, col].set_title(f'{title}\nscore {score:.2f}; relief {unit["central_relief_m"]:.0f} m', fontsize=10)
            if col == 4:
                fig.colorbar(im, ax=axes[row, :].tolist(), shrink=.8, pad=.01)
    fig.suptitle(f'{number:02d} {query["region"]} | {selected["selection_reason"]}\n600 m; shared row color limits; source grids north-up; no landform alignment', fontsize=11)
    save_figure(fig, f'pair_gallery_{number:02d}')
gallery = pd.DataFrame(gallery_records)
gallery.to_csv(OUT / 'gallery_panels.csv', index=False)
print('Rendered', len(selections), 'query galleries')

# Native labels of the Arizona candidates actually supplying matches

Count every distinct successful no-bare winner for strict 600 m queries under
each scenario. Report unique-candidate and query-use weighted summaries. This
describes selected source labels, not all Arizona patches or actual vegetation.

In [ ]:
CLASS_CODES = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
CLASS_LABELS = ['unknown', 'tree', 'shrub', 'grass', 'crop', 'built', 'bare/sparse', 'snow/ice', 'water', 'wetland', 'mangrove', 'moss/lichen']
native_paths = {
    'atacama_san_pedro': ROOT / 'data/raw/earth/worldcover/v200/2021/map/ESA_WorldCover_10m_2021_v200_S24W069_Map.tif',
    'northern_arizona': ROOT / 'data/raw/earth/worldcover/v200/2021/map/ESA_WorldCover_10m_2021_v200_N33W114_Map.tif',
    'wadi_rum': ROOT / screening_manifest['runs']['wadi_rum'] / 'earth/wadi_rum/worldcover_native_mosaic.tif',
}
use_rows = []
for scenario, prefix in [('baseline', 'b'), ('with_planarity', 'p')]:
    matched = az_free.loc[az_free[f'{scenario}_count'] > 0]
    for candidate_id, count in matched[f'{prefix}_candidate_id'].value_counts().items():
        use_rows.append({'scenario': scenario, 'candidate_id': candidate_id, 'query_uses': int(count)})
uses = pd.DataFrame(use_rows)
uses.to_csv(OUT / 'arizona_candidate_uses.csv', index=False)
native_ids = gallery_candidate_ids | set(uses['candidate_id'])


def native_counts(unit: dict[str, Any]) -> tuple[dict[str, int], str]:
    """Count native pixel centers inside the projected patch; no class resampling."""
    path = checked(native_paths[unit['region']])
    crs = grids[unit['region']]['crs_wkt']
    half = float(unit['size_m']) / 2
    x, y = float(unit['center_x_m']), float(unit['center_y_m'])
    bounds = (x - half, y - half, x + half, y + half)
    with rasterio.open(path) as src:
        geographic = transform_bounds(crs, src.crs, *bounds, densify_pts=21)
        floating = from_bounds(*geographic, transform=src.transform)
        left, top = int(np.floor(floating.col_off)), int(np.floor(floating.row_off))
        right = int(np.ceil(floating.col_off + floating.width))
        bottom = int(np.ceil(floating.row_off + floating.height))
        win = Window.from_slices((top, bottom), (left, right), boundless=True)
        values = src.read(1, window=win, boundless=True, fill_value=0)
        rr, cc = np.indices(values.shape)
        geo_x = src.transform.c + (left + cc + .5) * src.transform.a
        geo_y = src.transform.f + (top + rr + .5) * src.transform.e
        projected = project_points(src.crs, crs, geo_x.ravel().tolist(), geo_y.ravel().tolist())
        px, py = np.array(projected[0]), np.array(projected[1])
        selected = ((px >= bounds[0]) & (px < bounds[2]) & (py >= bounds[1]) & (py < bounds[3])).reshape(values.shape)
        assert np.isin(values[selected], CLASS_CODES).all() and selected.any()
        counts = {f'class_{code}_pixels': int(np.count_nonzero(values[selected] == code)) for code in CLASS_CODES}
        return counts, str(path.relative_to(ROOT))


native_rows = []
for unit_id in sorted(native_ids):
    unit = earth_lookup.loc[unit_id].to_dict()
    counts, native_source = native_counts(unit)
    total = sum(counts.values())
    native_rows.append({'candidate_id': unit_id, 'region': unit['region'], 'row': unit['row'], 'col': unit['col'],
                        'center_x_m': unit['center_x_m'], 'center_y_m': unit['center_y_m'], 'native_source': native_source,
                        'baseline_eligible': unit['baseline'], 'patch_mean_eligible': unit['patch_mean_bare80'],
                        'mean_30m_bare_fraction': unit['bare_mean'], 'native_pixels': total, **counts,
                        **{f'class_{code}_fraction': counts[f'class_{code}_pixels'] / total for code in CLASS_CODES}})
native = pd.DataFrame(native_rows)
native.to_csv(OUT / 'native_candidate_classes.csv', index=False)
joined = uses.merge(native, on='candidate_id', validate='many_to_one')
class_summary_rows = []
for scenario, group in joined.groupby('scenario'):
    for weighting in ['unique_candidate', 'query_use']:
        weights = np.ones(len(group)) if weighting == 'unique_candidate' else group['query_uses'].to_numpy()
        row = {'scenario': scenario, 'weighting': weighting, 'distinct_candidates': len(group),
               'matched_queries': int(group['query_uses'].sum()),
               'baseline_eligible_pct': float(100 * np.average(group['baseline_eligible'], weights=weights)),
               'patch_mean_eligible_pct': float(100 * np.average(group['patch_mean_eligible'], weights=weights))}
        row.update({f'class_{code}_pct': float(100 * np.average(group[f'class_{code}_fraction'], weights=weights)) for code in CLASS_CODES})
        class_summary_rows.append(row)
class_summary = pd.DataFrame(class_summary_rows)
class_summary.to_csv(OUT / 'arizona_class_summary.csv', index=False)
fig, ax = plt.subplots(figsize=(13, 5))
for offset, row in zip([-.3, -.1, .1, .3], class_summary.to_dict('records'), strict=True):
    ax.bar(np.arange(len(CLASS_CODES)) + offset, [row[f'class_{c}_pct'] for c in CLASS_CODES], .2,
           label=f'{row["scenario"]} / {row["weighting"]}')
ax.set(xticks=np.arange(len(CLASS_CODES)), xticklabels=CLASS_LABELS, ylabel='Mean native-pixel class fraction (%)', ylim=(0, 100),
       title='Arizona no-bare winners serving strict 600 m queries: source labels, not independent truth')
ax.tick_params(axis='x', labelrotation=30)
ax.legend(fontsize=8)
fig.tight_layout()
save_figure(fig, 'arizona_winner_classes')
print('Native classes inspected for', len(native), 'distinct Earth gallery/matched candidates')

# Findings and next decision

Descriptor coverage, selected source classes and spatial illustrations narrow
the next question. They do not validate analog relevance or authorize changing
a production screen. Keep the original baseline and the reserves untouched.

In [ ]:
# Read the Markdown notes below for the standalone report export.
report_notebook = json.loads((ROOT / 'notebooks/exploration/23_candidate_inspection_eda.ipynb').read_text(encoding="utf-8"))
report_notes = {entry["metadata"]["report_section"]: "".join(entry["source"])
                for entry in report_notebook["cells"]
                if entry["cell_type"] == "markdown" and "report_section" in entry["metadata"]}
report = [
    '# Candidate inspection: Moon and Earth',
    '',
    'This development-only inspection follows the [declared plan](../../exploration/23_candidate_inspection_eda.ipynb). Mars is excluded. Existing grids, masks, bands and lunar queries are unchanged. Detailed candidate diagnostics use eligible-at-size Earth pools; common-all-sizes results remain in notebook22.',
    *report_notes['narrative-cdd8f057-1'].split('\n'),
]
strict_summary = summary.loc[(summary['size_m'] == 600) & (summary['population'] == 'image_supported') & (summary['region'] == 'ALL_QUERY_WEIGHTED')]
for row in strict_summary.to_dict('records'):
    percentages = ' | '.join(f'{row[f]:.1f}%' for f in plot_fields)
    report.append(f'| {row["earth_site"]} / {row["policy"]} | {row["queries"]} | {percentages} |')
report += [
    *report_notes['narrative-cdd8f057-2'].split('\n'),
]
for row in class_summary.to_dict('records'):
    report.append(f'| {row["scenario"]} | {row["weighting"]} | {row["distinct_candidates"]} | {row["matched_queries"]} | '
                  f'{row["class_10_pct"]:.1f}% | {row["class_20_pct"]:.1f}% | {row["class_30_pct"]:.1f}% | {row["class_60_pct"]:.1f}% |')
report += [
    *report_notes['narrative-cdd8f057-3'].split('\n'),
]
for i, row in enumerate(selected_queries, 1):
    report.append(f'- [Gallery {i:02d}](pair_gallery_{i:02d}.png): {row["selection_reason"]}; selection group {row["group_queries"]} queries.')
report += [
    *report_notes['narrative-cdd8f057-4'].split('\n'),
    'This pass acquired no new imagery or elevation reference and did not inspect validation/final reserves. The [source-interpretation note](../screening-policy-20261029/SOURCE_INTERPRETATION.md) records the remaining WorldCover/FABDEM limitations. Findings and any changed next questions belong in the [living EDA plan](../../../EDA_PLAN.md).',
    '',
    '## Artifacts',
    '',
    '- [Pair diagnostics](candidate_diagnostics.csv.gz), [constraint coverage](constraint_coverage.csv), [region macro coverage](constraint_coverage_region_macro.csv).',
    '- [Selected queries](gallery_queries.csv), [georeferenced gallery panels](gallery_panels.csv).',
    '- [Native class counts](native_candidate_classes.csv), [Arizona candidate reuse](arizona_candidate_uses.csv), [class summaries](arizona_class_summary.csv).',
    '- [Constraint support](constraint_support.png), [descriptor arrangements](descriptor_arrangements.png), [winner classes](arizona_winner_classes.png).',
    '- [Provenance](provenance.json).',
    '',
]
(OUT / 'REPORT.md').write_text('\n'.join(report), encoding='utf-8')
provenance['inputs'] = evidence
provenance['outputs'] = {str(p.relative_to(OUT)): file_hash(p) for p in sorted(OUT.rglob('*')) if p.is_file() and p.name != 'provenance.json'}
provenance['status'] = 'complete'
(OUT / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
display(Markdown('\n'.join(report)))
print('Complete:', OUT)


## 600 m strict-lunar constraint support

Percentages mean queries with at least one candidate meeting the listed requirements. Different candidates may provide each marginal match. Dropping a band is an explanatory diagnostic, not an adopted screening or matching rule.

| Earth site / policy | Queries | Median slope | P90 slope | Relief | Median + P90 | Median + relief | P90 + relief | All three | + Planarity |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |


All sizes, both lunar views, individual lunar regions and equal-region macro means are exported. The prior notebook22 per-site best scores and in-band counts are reproduced for every audited pool/query/scenario. No model, normalization or relevance label was fitted.

## Arizona candidates actually supplying strict 600 m matches

Native WorldCover counts cover every distinct successful no-bare winner under each score scenario. The unique-candidate average gives each selected patch one vote; query-use weighting counts repeated selection. Neither represents all Arizona terrain or independent observations.

| Scenario | Weighting | Distinct candidates | Matched queries | Tree labels | Shrub labels | Grass labels | Bare/sparse labels |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |


These are mean native-pixel label fractions, not measured vegetation percentages or exactly the area-weighted fractions on the analysis grid. Source labels are not independently verified current cover; a plausible estimated ground surface does not establish a usable outdoor training surface.

## Spatial inspection

Each figure compares a fixed lunar query with Atacama/Wadi baseline-policy winners and Arizona no-bare winners under baseline and planarity-aware scoring. Shared physical color limits within each row keep amplitude visible. All patches retain their source-grid orientation; no rotations, landform alignment or image similarity score is introduced. A score above one is explicitly outside the declared bands.

Example selection is conditional on descriptor results and documented in the selection table. It is not a blind or representative assessment of analog accuracy. Repeated Earth candidates across queries remain identifiable. The maps can motivate source checks; they cannot establish that a DEM feature is real.



## Decision and next evidence

Preserve the baseline screen and keep diagnostic alternatives separate. Use component failures and spatial arrangements to distinguish missing descriptor combinations from source or land-cover concerns. The next bounded source check should target the newly admitted candidates actually serving queries, with independent imagery/reference terrain where available. Mark unresolved cases uncertain rather than inventing relevance labels.
